# 02 - Feature Engineering Prototyping

Prototypes and visualizes the derived features implemented in
`src/data_pipeline/feature_engineering.py`, and validates a raw batch
against the Great Expectations suite used by
`src/data_pipeline/validate.py` before they are applied.

Each derivation below is a pure function so it can be unit tested in
isolation (see `tests/unit/test_feature_engineering.py`) and reused
identically at training time (`src/data_pipeline/transform.py`) and
inference time (`src/deployment/scoring/score.py`).

In [ ]:
import sys
from datetime import date
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_pipeline.feature_engineering import (
    derive_age_band,
    derive_comorbidity_score,
    derive_los_bucket,
    derive_polypharmacy_flag,
    derive_prior_utilization_rate,
    engineer_features,
)
from src.data_pipeline.ingest import generate_synthetic_encounters
from src.data_pipeline.transform import clean_raw_encounters, transform
from src.data_pipeline.validate import validate_dataframe

pd.set_option("display.max_columns", None)

In [ ]:
raw = generate_synthetic_encounters(n_rows=5000, start_date=date(2024, 1, 1), seed=42)
raw.shape

## 1. Validate the raw batch

Runs the same `encounters_suite` expectation suite the pipeline uses
to gate bad data before it reaches feature engineering.

In [ ]:
validation_result = validate_dataframe(raw)
validation_result["success"]

## 2. Prototype individual derivations

In [ ]:
age_band = derive_age_band(raw["age"])
age_band.value_counts().sort_index()

In [ ]:
los_bucket = derive_los_bucket(raw["length_of_stay"])
los_bucket.value_counts().sort_index()

In [ ]:
comorbidity_score = derive_comorbidity_score(raw["comorbidity_count"], raw["charlson_index"])
prior_utilization_rate = derive_prior_utilization_rate(raw["prior_admissions_12mo"], raw["prior_ed_visits_12mo"])
polypharmacy_flag = derive_polypharmacy_flag(raw["num_medications"])

pd.DataFrame(
    {
        "comorbidity_score": comorbidity_score,
        "prior_utilization_rate": prior_utilization_rate,
        "polypharmacy_flag": polypharmacy_flag,
    }
).describe()

## 3. Full transform: clean -> engineer_features

`transform()` chains `clean_raw_encounters()` (dedup, date coercion,
drop rows missing the label) with `engineer_features()`.

In [ ]:
cleaned = clean_raw_encounters(raw)
features = engineer_features(cleaned, keep_ethnicity=True)

# Equivalent to:
# features = transform(raw)

features.head()

In [ ]:
features.dtypes

## 4. Sanity-check derived features against the label

Each derived feature should show a monotonic-ish relationship with
`readmitted_30d` — if not, revisit the derivation or the bucket
boundaries in `src/data_pipeline/feature_engineering.py`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features.groupby("age_band")["readmitted_30d"].mean().plot.bar(ax=axes[0], title="Readmission rate by age_band")
features.groupby("los_bucket")["readmitted_30d"].mean().plot.bar(ax=axes[1], title="Readmission rate by los_bucket")
features.groupby("polypharmacy_flag")["readmitted_30d"].mean().plot.bar(ax=axes[2], title="Readmission rate by polypharmacy_flag")

for ax in axes:
    ax.set_ylabel("readmission rate")

fig.tight_layout()
plt.show()

In [ ]:
comorbidity_bins = pd.cut(features["comorbidity_score"], bins=5)
features.groupby(comorbidity_bins)["readmitted_30d"].mean().plot.bar(
    title="Readmission rate by comorbidity_score bin", figsize=(8, 4)
)
plt.ylabel("readmission rate")
plt.show()

## Next steps

Continue to `03_model_experiments.ipynb` to train and compare
candidate models on this feature table.